# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

This playbook turns my validated Lane-4 output into a **human-reviewed content action queue**: a
ranked shortlist of visible pages that are *observed* to under-capture clicks for their search
position, each carrying a reason a reviewer can trust. It is **decision-support**, not automation —
the model and rule surface candidates; a person decides and edits.

**Score.** `gap * log1p(impressions_90d)`, where `gap = max(tier_median_ctr - ctr, 0)`. Big
shortfalls on high-traffic pages rank first. Transparent on purpose (my Week-6 audit showed the
learned model barely beats this tier-gap on unseen clients, so the shippable scorer stays the
readable one; the model is kept as a cross-check).

**Archetype -> action mapping.**

| Archetype | Meaning | Action | Reason code |
|---|---|---|---|
| `high_value_ctr_gap` | strong position, high traffic, CTR < half its tier median | **priority review** title/meta/snippet | `high_traffic_low_ctr_strong_position` |
| `ctr_gap_candidate` | visible, CTR below half its tier median | review title/meta/snippet | `low_ctr_for_position_tier` |
| `bottom_of_page_one` | page-one but position > 7 | monitor only (shortfall partly expected for rank) | `position_expected_shortfall` |
| `low_volume_watch` | under 500 impressions | monitor only (too noisy to act) | `insufficient_volume` |
| `healthy_ctr` | CTR at or above tier median | protect / no action | `meets_position_expectation` |

**Decay / refresh insight.** CTR-opportunity is a *snippet* fix, separate from content decay. But
pages that are **both** a CTR gap **and** in the content-decay age window (271-365 days, the paper's
Finding #2) are the strongest combined bets: pair the metadata review with a content refresh. The
code flags these with `in_decay_window`.

In [1]:
# Setup + build the ranked action queue with archetypes, reason codes, and actions.
import os, sys, json, subprocess
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git","clone","--depth","1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        "flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
vis = df[(df["impressions_90d"] >= 100) & (df["ctr"].notna())].copy()
vis["tier_median_ctr"] = vis.groupby("position_tier")["ctr"].transform("median")
vis["gap"] = (vis["tier_median_ctr"] - vis["ctr"]).clip(lower=0)

def archetype(r):
    if r["ctr"] >= r["tier_median_ctr"]:                                   return "healthy_ctr"
    if r["impressions_90d"] < 500:                                         return "low_volume_watch"
    if r["avg_position"] > 7 and r["position_tier"] == "page_1":           return "bottom_of_page_one"
    if r["impressions_90d"] >= 3000 and r["ctr"] < 0.5*r["tier_median_ctr"]: return "high_value_ctr_gap"
    if r["ctr"] < 0.5*r["tier_median_ctr"]:                                return "ctr_gap_candidate"
    return "minor_ctr_gap"
vis["archetype"] = vis.apply(archetype, axis=1)

ACTION = {"high_value_ctr_gap": ("priority_review_title_meta", "high_traffic_low_ctr_strong_position"),
          "ctr_gap_candidate":  ("review_title_meta",          "low_ctr_for_position_tier")}
queue = vis[vis["archetype"].isin(ACTION)].copy()
queue["action"]      = queue["archetype"].map(lambda a: ACTION[a][0])
queue["reason_code"] = queue["archetype"].map(lambda a: ACTION[a][1])
queue["score"]       = queue["gap"] * np.log1p(queue["impressions_90d"])
queue["confidence"]  = np.where(queue["impressions_90d"] >= 3000, "high", "medium")
queue["in_decay_window"] = ((queue["content_age_days"] >= 271) & (queue["content_age_days"] <= 365))
queue = queue.sort_values("score", ascending=False).reset_index(drop=True)

print(f"actionable queue: {len(queue):,} pages | in content-decay window: {int(queue['in_decay_window'].sum())}")
print("\narchetype -> action counts:")
print(queue["archetype"].value_counts().to_string())
print("\nTop 8 of the queue:")
print(queue[["content_id","position_tier","impressions_90d","avg_position","ctr",
             "tier_median_ctr","action","confidence","in_decay_window"]].head(8).round(3).to_string(index=False))

actionable queue: 3,237 pages | in content-decay window: 659

archetype -> action counts:
archetype
ctr_gap_candidate     2299
high_value_ctr_gap     938

Top 8 of the queue:
          content_id position_tier  impressions_90d  avg_position  ctr  tier_median_ctr                     action confidence  in_decay_window
content_0919dd345d80        page_1           119217           7.0 0.02             0.23 priority_review_title_meta       high             True
content_d274ac4158ef        page_1            65138           6.8 0.01             0.23 priority_review_title_meta       high             True
content_e5f459e737b7        page_1            56363           5.9 0.01             0.23 priority_review_title_meta       high            False
content_339b357d04c7        page_1            46879           3.7 0.01             0.23 priority_review_title_meta       high            False
content_65114d89496d        page_1            72631           6.5 0.02             0.23 priority_review_title_

## 2. Intended use and limits

**Who uses it, for what.** A content or SEO reviewer with limited weekly capacity uses the ranked
queue to decide *which pages to open first* for a title / meta / snippet review. It is a triage aid
for human editors.

**Where it stops being valid.** It is built on one anonymized 90-day snapshot, so it is
**observational and directional**. The score says a page under-captures clicks *relative to peers at
its position in this sample* — it does **not** predict how many clicks an edit would win, and it does
not generalize as a precise predictor to unseen clients (my Week-6 client-grouped audit showed only
directional accuracy there). Feature and target share the 90-day window, so this is opportunity
**scoring**, not future prediction.

**Cost / value thinking.** The value framing is *exposure at stake* — the impressions sitting on
under-performing pages — not promised clicks. The code below sizes a realistic reviewer capacity
against the queue so the plan stays practical.

In [2]:
# Cost/value: size the queue against a realistic review capacity, and the exposure at stake.
inventory = len(vis)
q = len(queue)
for cap in (20, 50, 100):
    top = queue.head(cap)
    exposure = int(top["impressions_90d"].sum())
    print(f"review capacity {cap:>3}: covers {cap/q*100:4.1f}% of the queue | "
          f"exposure at stake in those {cap} pages: {exposure:,} impressions/90d")
print(f"\nqueue is {q:,} of {inventory:,} visible pages ({q/inventory*100:.1f}%). Value is framed as")
print("EXPOSURE a reviewer could act on, not clicks promised -- edits are hypotheses tested after.")

review capacity  20: covers  0.6% of the queue | exposure at stake in those 20 pages: 1,020,484 impressions/90d
review capacity  50: covers  1.5% of the queue | exposure at stake in those 50 pages: 2,015,419 impressions/90d
review capacity 100: covers  3.1% of the queue | exposure at stake in those 100 pages: 3,144,711 impressions/90d

queue is 3,237 of 22,006 visible pages (14.7%). Value is framed as
EXPOSURE a reviewer could act on, not clicks promised -- edits are hypotheses tested after.


## 3. Human review + the no-go list

**Every recommendation is a prompt for a human, never an automatic edit.** Before acting, a reviewer
checks: is the low CTR explained by a brand/navigational query, a SERP feature, or intent mismatch
rather than a weak title? Is the volume real (not a handful of impressions)? Does the page's actual
within-tier position justify some of the shortfall?

**What must NOT be automated (no-go list):**

- **No auto-editing** of titles or meta — humans write and approve every change.
- **No action on `low_volume_watch` or `bottom_of_page_one`** archetypes — kept out of the actionable
  queue on purpose (noise and position-expected shortfall).
- **No action on zero-click, high-impression pages** without a human check — often tracking,
  attribution, or brand-navigational effects, not a snippet a rewrite can fix.
- **No causal claims** ("this refresh will recover traffic") and **no claims about Google's algorithm
  or AI rankings.**
- **No client-identifying output** — pseudonymized IDs and aggregates only.

In [3]:
# Quantify the no-go / held-back segments (excluded from the actionable queue by design).
held = vis[vis["archetype"].isin(["low_volume_watch","bottom_of_page_one","minor_ctr_gap"])]
print("Held OUT of the actionable queue on purpose:")
print(held["archetype"].value_counts().to_string())
# zero-click high-impression pages need a human check even inside the queue
ambiguous = queue[(queue["ctr"] == 0) & (queue["impressions_90d"] >= 5000)]
print(f"\nInside the queue but flagged for mandatory human check "
      f"(zero clicks at >=5k impressions): {len(ambiguous)} pages")
print("These are surfaced with a warning, never auto-actioned.")

Held OUT of the actionable queue on purpose:
archetype
low_volume_watch      3148
minor_ctr_gap         2315
bottom_of_page_one    1607

Inside the queue but flagged for mandatory human check (zero clicks at >=5k impressions): 68 pages
These are surfaced with a warning, never auto-actioned.


## 4. Monitoring / retrain triggers

Light, practical signals that the recommendations have gone stale:

- **Flagged-share drift.** If the actionable queue's share of visible pages moves materially from the
  recorded baseline (below), the input data or SERP landscape likely shifted — investigate before
  trusting the queue.
- **Tier-median shift.** If position-tier median CTRs move (Google SERP or layout changes), recompute
  the tier baselines the score depends on.
- **New clients / content types** not represented in the current snapshot — re-run the Week-6 audit
  before extending the playbook to them.
- **Cadence.** Regenerate quarterly on a fresh snapshot. Retrain the expected-CTR cross-check model
  only if its client-grouped MAE degrades against the recorded receipt — and remember it currently
  only ties the transparent rule, so complexity is not yet justified for production.

In [4]:
# Record the monitoring baseline (the reference future runs are compared against).
monitor_baseline = {
    "flagged_share_of_visible": round(len(queue)/len(vis), 3),
    "queue_size": int(len(queue)),
    "tier_median_ctr": {k: round(float(v), 3) for k, v in
                        vis.groupby("position_tier")["ctr"].median().items()},
    "decay_window_overlap": int(queue["in_decay_window"].sum()),
}
print("Monitoring baseline (compare future runs to this):")
print(json.dumps(monitor_baseline, indent=2))

Monitoring baseline (compare future runs to this):
{
  "flagged_share_of_visible": 0.147,
  "queue_size": 3237,
  "tier_median_ctr": {
    "deep": 0.0,
    "page_1": 0.23,
    "page_3_5": 0.06,
    "striking": 0.15,
    "top_3": 0.19
  },
  "decay_window_overlap": 659
}


## 5. Exports for the paper

Three artifacts my paper will build on next week: the ranked queue (regenerated each run, kept out of
git by the leak-guard), two reusable figures committed to `work/figures/`, and a metrics JSON receipt
committed to `work/outputs/`.

In [5]:
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# (1) ranked queue CSV -> work/outputs/ (data file; stays out of git, regenerated each run)
cols = ["content_id","client_id","content_type","position_tier","impressions_90d","avg_position",
        "ctr","tier_median_ctr","gap","score","archetype","reason_code","action","confidence","in_decay_window"]
queue[cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)

# (2) figures -> work/figures/ (committed, reused in the paper)
fig, ax = plt.subplots(figsize=(6,3.2))
order = ["top_3","striking","page_1","page_3_5","deep"]
tm = vis.groupby("position_tier")[["ctr","tier_median_ctr"]].mean().reindex([o for o in order if o in vis["position_tier"].unique()])
tm["ctr"].plot(kind="bar", ax=ax, color="#c0392b", label="mean actual CTR")
ax.set_title("Actual CTR by position tier (opportunity sits below the tier norm)")
ax.set_ylabel("CTR (x100)"); ax.set_xlabel("position tier"); ax.legend()
fig.tight_layout(); fig.savefig("work/figures/ctr_by_position_tier.png", dpi=120); plt.close(fig)

fig, ax = plt.subplots(figsize=(6,3.2))
queue["archetype"].value_counts().plot(kind="bar", ax=ax, color="#2c3e50")
ax.set_title("Content action queue composition by archetype")
ax.set_ylabel("pages"); ax.set_xlabel("")
fig.tight_layout(); fig.savefig("work/figures/queue_by_archetype.png", dpi=120); plt.close(fig)

# (3) metrics receipt -> work/outputs/ (committed)
json.dump({"queue_size": int(len(queue)),
           "flagged_share_of_visible": round(len(queue)/len(vis), 3),
           "decay_window_overlap": int(queue["in_decay_window"].sum()),
           "archetype_counts": queue["archetype"].value_counts().to_dict(),
           "monitoring_baseline": monitor_baseline},
          open("work/outputs/playbook_metrics.json", "w"), indent=2)

print("exports written:")
print("  work/outputs/action_playbook_queue.csv   (queue; kept out of git, regenerated each run)")
print("  work/figures/ctr_by_position_tier.png     (commit)")
print("  work/figures/queue_by_archetype.png       (commit)")
print("  work/outputs/playbook_metrics.json        (commit; the paper's receipts)")

exports written:
  work/outputs/action_playbook_queue.csv   (queue; kept out of git, regenerated each run)
  work/figures/ctr_by_position_tier.png     (commit)
  work/figures/queue_by_archetype.png       (commit)
  work/outputs/playbook_metrics.json        (commit; the paper's receipts)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.